# Mastra + MongoDB Atlas: 2. Bring Your Own Operational Collections

Notebook 01 searched `sample_mflix.embedded_movies`, a collection nobody in this
repository created, with its own field names and its own Atlas index. That worked because
of two arguments passed almost in passing: `collectionName` and `searchIndexName`.

This notebook is about that path properly. It is the difference between a vector store
you hand your data to and a vector store you point at data you already have.

## What you'll learn
1. Why a vector store that owns its collections is the wrong shape for existing data
2. Register a collection you own with `collectionName` + `searchIndexName`
3. Why a bring-your-own index refuses writes, and what `allowWrites` opts into
4. What the `__mastra_vector_indexes__` registry stores and why it has to be durable
5. Why `deleteIndex` is safe on a BYO collection, and what it still deletes

## Prerequisites
- Notebook 01 completed (it explains configuration, embeddings, and `query`)
- A free M0 cluster is enough, but see the note on the index budget below

## The problem

Say you have a `movie_reviews` collection. Your application writes to it, your
dashboards read from it, and its schema is whatever your app decided years ago. Now you
want semantic search over the review text.

The obvious approach with most vector stores is to copy: read the reviews, embed them,
write vectors plus a copy of the text into whatever collection the vector store manages.
That gives you two problems immediately.

**Duplication.** The review text now lives in two places.

**Drift.** When a review is edited or deleted, the operational collection changes and the
vector copy does not, until you build and maintain a sync pipeline. You have taken on an
ongoing correctness obligation to get search.

MongoDB does not need the copy. The vectors can live in the review documents themselves,
as one more field, and `$vectorSearch` queries them where they are. There is one copy of
the review, and it is the one your application already writes.

What that requires from the vector store is restraint: it must query a collection it does
not own, must not assume the field names, and must not write to it. That is what
`MongoDBVector`'s bring-your-own mode is.

In [1]:
import { loadNotebookConfig } from './src/config.js';
import { connect, createAtlasEmbedder } from './src/utils.js';

const cfg = loadNotebookConfig();
console.log('Configuration:', cfg.summary);

const { client } = await connect(cfg.mongodbUri);
const db = client.db(cfg.dbName);

Configuration: db=sample_mflix · embeddings=voyage-3-large (2048-d) · llm=claude-haiku-4-5 via gateway


## 1. An operational collection

To have something we genuinely own, we build it. These twelve documents stand in for the
collection your application already writes: ObjectId `_id`s, application field names, no
concession to any vector store.

Note the embedding dimension. Notebook 01 used 2048 because that is what `sample_mflix`
stores. Here we choose 1024: dimension is a property of the index, not of your cluster or
your account, and a shorter vector is cheaper to store and faster to compare. Voyage's
`voyage-3-large` is trained so that its shorter outputs remain usable, which is what makes
the choice a real one.

In [2]:
const REVIEWS = 'movie_reviews';

const reviewTexts = [
  { movie: 'Blade Runner', author: 'ada',   rating: 5, body: 'The rain-soaked city does more character work than the dialogue. Every frame is exhausted and beautiful.' },
  { movie: 'Blade Runner', author: 'lin',   rating: 3, body: 'Gorgeous but glacial. I admired it more than I enjoyed it.' },
  { movie: 'The Thing',    author: 'ada',   rating: 5, body: 'Paranoia as a special effect. Nobody trusts anybody and the practical creature work still holds up.' },
  { movie: 'The Thing',    author: 'omar',  rating: 4, body: 'A locked-room mystery where the room is Antarctica. Genuinely tense.' },
  { movie: 'Arrival',      author: 'lin',   rating: 5, body: 'A first-contact film about grammar and grief. The structure is the whole point.' },
  { movie: 'Arrival',      author: 'ada',   rating: 4, body: 'Quiet, sad, and smarter than it needs to be. The linguistics is almost plausible.' },
  { movie: 'Mad Max: Fury Road', author: 'omar', rating: 5, body: 'Two hours of vehicular carnage with more coherent staging than most drama.' },
  { movie: 'Mad Max: Fury Road', author: 'lin',  rating: 4, body: 'Relentless. I needed a lie down afterwards, which I think was intended.' },
  { movie: 'Paddington 2', author: 'ada',   rating: 5, body: 'Sincere, warm, and impeccably constructed. The prison sequence is a farce done properly.' },
  { movie: 'Paddington 2', author: 'omar',  rating: 5, body: 'The kindest film about a bear and a marmalade sandwich ever made.' },
  { movie: 'Heat',         author: 'lin',   rating: 4, body: 'The bank robbery is the best shootout ever filmed. The rest is two men being sad at each other.' },
  { movie: 'Heat',         author: 'omar',  rating: 5, body: 'A heist film that cares more about the professionals than the money.' },
];

// Start clean so re-running the notebook is idempotent.
await db.collection(REVIEWS).drop().catch(() => {});

const embedder = createAtlasEmbedder({
  apiKey: cfg.atlasApiKey,
  model: 'voyage-3-large',
  dimensions: 1024,        // this collection's choice, independent of notebook 01's 2048
});

const bodyVectors = await embedder.embedDocuments(reviewTexts.map((r) => r.body));

await db.collection(REVIEWS).insertMany(
  reviewTexts.map((review, i) => ({
    movie_title: review.movie,     // our field names, not Mastra's
    author: review.author,
    rating: review.rating,
    body: review.body,
    body_embedding: bodyVectors[i],
    created_at: new Date('2026-01-15T00:00:00Z'),
  })),
);

console.log(`Inserted ${await db.collection(REVIEWS).countDocuments()} reviews`);
const first = await db.collection(REVIEWS).findOne({}, { projection: { body_embedding: 0 } });
console.log('Field names:', Object.keys(first ?? {}).join(', '));
console.log('_id type   :', first?._id.constructor.name);

Inserted 12 reviews


Field names: _id, movie_title, author, rating, body, created_at


_id type   : ObjectId


The `_id` is an `ObjectId`, not a string. That is worth flagging because Mastra's
`QueryResult.id` is typed `string`, so the store coerces on the way out and matches both
forms on the way in. It is the kind of detail that only shows up against real operational
data, and it is handled.

## 2. The Atlas index, created by us

The vector index is also ours. In a real system this is a migration, a Terraform
resource, or an Atlas UI click: something in your infrastructure, not something your
application framework does at startup.

We declare `movie_title` and `rating` as filter fields for the pushdown path from
notebook 01, and `_id` so the pre-filter fallback also works. Notebook 01 hit exactly this:
without `_id`, filtering on an undeclared field fails with
`Path '_id' needs to be indexed as filter`.

> **M0 index budget.** A free cluster allows three Atlas Search indexes. `sample_mflix`
> already carries two (`movies_vector_index`, `movies_fulltext_index`), so this is the
> third and last. The cleanup section drops it. If `createSearchIndex` reports a limit
> here, that is why.

In [3]:
const REVIEWS_INDEX = 'movie_reviews_vector_index';

await db.collection(REVIEWS).createSearchIndex({
  name: REVIEWS_INDEX,
  type: 'vectorSearch',
  definition: {
    fields: [
      { type: 'vector', path: 'body_embedding', numDimensions: 1024, similarity: 'cosine' },
      { type: 'filter', path: 'movie_title' },
      { type: 'filter', path: 'rating' },
      // Declaring _id lets the pre-filter fallback work for undeclared fields.
      { type: 'filter', path: '_id' },
    ],
  },
} as any);

// Atlas builds search indexes asynchronously: PENDING -> BUILDING -> READY. Querying
// before it is queryable returns an error, so poll.
interface SearchIndexStatus { name: string; status?: string; queryable?: boolean }

for (let attempt = 1; attempt <= 60; attempt++) {
  const found = await db.collection(REVIEWS).listSearchIndexes(REVIEWS_INDEX).toArray();
  // The driver types listSearchIndexes as `{ name: string }[]`; the documents it
  // actually returns carry `status` and `queryable` too.
  const status = found[0] as SearchIndexStatus | undefined;
  if (status?.queryable) {
    console.log(`Index queryable after ~${attempt * 5}s (status ${status.status})`);
    break;
  }
  await new Promise((resolve) => setTimeout(resolve, 5000));
}

Index queryable after ~40s (status READY)


## 3. Registering it with Mastra

Now the actual bring-your-own step. Three things make it work:

- **`embeddingFieldPath`** on the constructor: our vectors are in `body_embedding`.
  This is per-store, not per-index, which is why a store pointed at 2048-d movie plots
  and a store pointed at 1024-d review bodies are two `MongoDBVector` instances.
- **`collectionName`** on `createIndex`: the collection to read. Passing this is what
  marks the index bring-your-own, and from then on the store never creates or drops that
  collection.
- **`searchIndexName`**: the Atlas index to query. Without it Mastra would look for
  `movie_reviews_vector_index`... which happens to be exactly what we named it, but
  relying on that coincidence would be a poor lesson.

`createIndex` here provisions nothing. It records a mapping.

In [4]:
import { MongoDBVector } from '@mastra/mongodb';
import { APP_NAME } from './src/utils.js';

const reviews = new MongoDBVector({
  id: 'reviews',
  uri: cfg.mongodbUri,
  dbName: cfg.dbName,
  options: { appName: APP_NAME },
  embeddingFieldPath: 'body_embedding',   // our field, not the `embedding` default
});
await reviews.connect();

await reviews.createIndex({
  indexName: 'reviews',
  dimension: 1024,
  metric: 'cosine',
  collectionName: REVIEWS,                // <- this is what makes it BYO
  searchIndexName: REVIEWS_INDEX,
  filterFields: ['movie_title', 'rating'],
});

console.log('Logical indexes:', await reviews.listIndexes());
console.log('describeIndex  :', await reviews.describeIndex({ indexName: 'reviews' }));

Logical indexes: [ 'mflix_plots', 'reviews' ]


describeIndex  : { dimension: 1024, count: 12, metric: 'cosine' }


`describeIndex` reports 12: it counted the documents already in our collection. Nothing
was ingested, copied, or migrated. The vectors were already in the right place.

Querying works exactly as in notebook 01, with `metadataMode: 'document'` because our
documents have their own shape.

In [5]:
const semantic = await reviews.query({
  indexName: 'reviews',
  queryVector: await embedder.embedQuery('a film that is tense and full of distrust'),
  topK: 4,
  metadataMode: 'document',
});

for (const hit of semantic) {
  const review = hit.metadata as { movie_title?: string; author?: string; body?: string };
  console.log(`${hit.score.toFixed(4)}  ${String(review.movie_title).padEnd(18)} @${review.author}`);
  console.log(`        ${review.body}`);
}

0.7299  The Thing          @ada


        Paranoia as a special effect. Nobody trusts anybody and the practical creature work still holds up.


0.7132  Paddington 2       @ada


        Sincere, warm, and impeccably constructed. The prison sequence is a farce done properly.


0.7128  The Thing          @omar


        A locked-room mystery where the room is Antarctica. Genuinely tense.


0.7027  Blade Runner       @ada


        The rain-soaked city does more character work than the dialogue. Every frame is exhausted and beautiful.


Reviews about paranoia and distrust come back first, and none of them contain the word
"distrust". The filter pushdown from notebook 01 works here too, against our own field
names.

In [6]:
const highlyRated = await reviews.query({
  indexName: 'reviews',
  queryVector: await embedder.embedQuery('beautiful to look at'),
  topK: 3,
  filter: { rating: { $gte: 5 } },   // declared filter field -> pushed down
  metadataMode: 'document',
});

for (const hit of highlyRated) {
  const review = hit.metadata as { movie_title?: string; rating?: number };
  console.log(`${hit.score.toFixed(4)}  ${String(review.movie_title).padEnd(18)} rating ${review.rating}`);
}

0.7463  Blade Runner       rating 5


0.7187  Paddington 2       rating 5


0.7030  Arrival            rating 5


## 4. Read-only by default

Here is the part of this design worth arguing about, and the reason it defaults the way
it does.

`MongoDBVector` has `upsert`, `updateVector`, `deleteVector`, and `deleteVectors`. On a
collection Mastra created, all four are fine. It owns those documents. On *your*
operational collection they are a different proposition: `deleteVectors` with a filter
against a live production collection deletes production data.

So a bring-your-own index is **read-only unless you say otherwise**. Not read-only by
convention or by documentation. The write methods refuse.

In [7]:
const attempts: Array<[string, () => Promise<unknown>]> = [
  ['upsert', () => reviews.upsert({
    indexName: 'reviews',
    vectors: [bodyVectors[0]!],
    metadata: [{ note: 'injected' }],
  })],
  ['deleteVector', () => reviews.deleteVector({
    indexName: 'reviews',
    id: String(first?._id),
  })],
];

for (const [name, attempt] of attempts) {
  try {
    await attempt();
    console.log(`${name}: SUCCEEDED -- unexpected`);
  } catch (error) {
    console.log(`${name}: refused -- ${(error as Error).message.split('\n')[0]}`);
  }
}

console.log(`\nDocument count unchanged: ${await db.collection(REVIEWS).countDocuments()}`);

upsert: refused -- Index "reviews" targets the bring-your-own collection "movie_reviews", which is read-only by default — Mastra does not modify or delete caller-owned operational documents. To allow writes, opt in explicitly: createIndex({ indexName: "reviews", collectionName: "movie_reviews", allowWrites: true, ... }).


deleteVector: refused -- Index "reviews" targets the bring-your-own collection "movie_reviews", which is read-only by default — Mastra does not modify or delete caller-owned operational documents. To allow writes, opt in explicitly: createIndex({ indexName: "reviews", collectionName: "movie_reviews", allowWrites: true, ... }).



Document count unchanged: 12


Both refused, and the count is still 12. The default is the safe one, which is the right
way round: a reader who never thinks about this cannot lose data by not thinking about it.

`allowWrites: true` opts in. It belongs on an index whose collection you are deliberately
letting Mastra manage, a collection built for retrieval that your app does not otherwise
write to.

In [8]:
// Re-registering the same logical index with allowWrites flips the flag. Idempotent:
// it updates the registry entry rather than provisioning anything.
await reviews.createIndex({
  indexName: 'reviews',
  dimension: 1024,
  metric: 'cosine',
  collectionName: REVIEWS,
  searchIndexName: REVIEWS_INDEX,
  filterFields: ['movie_title', 'rating'],
  allowWrites: true,
});

const [newVector] = await embedder.embedDocuments([
  'A quiet film about language and loss that rewards a second viewing.',
]);

const [writtenId] = await reviews.upsert({
  indexName: 'reviews',
  vectors: [newVector!],
  metadata: [{ movie_title: 'Arrival', author: 'mastra', rating: 5 }],
  documents: ['A quiet film about language and loss that rewards a second viewing.'],
});

console.log('Wrote id:', writtenId);
console.log('Document count now:', await db.collection(REVIEWS).countDocuments());

Wrote id: bd9db981-0805-4168-9876-5fc955a42727


Document count now: 13


The write succeeded. Now look at what it actually wrote, because this is the detail that
matters when you opt in.

In [9]:
const written = await db.collection(REVIEWS).findOne({ _id: writtenId as any });
const writtenEmbedding = written?.body_embedding as number[] | undefined;

console.log('Mastra-written document (embedding replaced with its length):');
console.log(JSON.stringify({ ...written, body_embedding: `<${writtenEmbedding?.length}-d vector>` }, null, 2));

console.log('\nAn application-written document, for comparison:');
console.log(JSON.stringify(
  await db.collection(REVIEWS).findOne({ author: 'ada' }, { projection: { body_embedding: 0 } }),
  null,
  2,
));

Mastra-written document (embedding replaced with its length):


{
  "_id": "bd9db981-0805-4168-9876-5fc955a42727",
  "body_embedding": "<1024-d vector>",
  "document": "A quiet film about language and loss that rewards a second viewing.",
  "metadata": {
    "movie_title": "Arrival",
    "author": "mastra",
    "rating": 5
  }
}



An application-written document, for comparison:


{
  "_id": "6a698e7b2f3c8a0840e4f37e",
  "movie_title": "Blade Runner",
  "author": "ada",
  "rating": 5,
  "body": "The rain-soaked city does more character work than the dialogue. Every frame is exhausted and beautiful.",
  "created_at": "2026-01-15T00:00:00.000Z"
}


The two documents do not have the same shape. Mastra wrote a string `_id`, put the text in
`document`, and nested the fields we passed under `metadata`, because that is the schema
its managed mode uses. Our own reviews have `movie_title` and `body` at the top level.

The vector did land in the right place. `embeddingFieldPath` is respected, so
`body_embedding` holds a 1024-d vector and this document is genuinely searchable, but it
is a document your application's own readers do not understand.

That is the honest cost of `allowWrites` on a collection with an established schema, and
it is a good argument for the default. Two workable positions:

- **Read-only BYO** (the default): your application owns writes, including the embedding
  field, and Mastra only ever queries.
- **`allowWrites` on a collection built for Mastra**: a retrieval collection where the
  managed shape *is* the schema, and nothing else reads it.

Mixing writers on one collection with two schemas is the position to avoid. Let us undo
it. `deleteVector` is permitted now that writes are allowed.

In [10]:
await reviews.deleteVector({ indexName: 'reviews', id: writtenId! });
console.log('Removed. Document count back to:', await db.collection(REVIEWS).countDocuments());

Removed. Document count back to: 12


## 5. The registry

`allowWrites`, `collectionName`, and `searchIndexName` were passed to `createIndex`: a
call made once, in one process. Every later `query` and every refused `upsert` needs to
know them.

Keeping them in memory would mean a restarted process forgets which of its indexes are
bring-your-own. That failure is not cosmetic: `deleteIndex` behaves differently for BYO
and managed indexes, and an index misclassified as managed after a restart is one whose
collection could be dropped. So the mapping is durable, in a collection Mastra owns.

In [11]:
for (const entry of await db.collection('__mastra_vector_indexes__').find({}).toArray()) {
  console.log(JSON.stringify(entry));
}

{"_id":"mflix_plots","allowWrites":false,"collectionName":"embedded_movies","dimension":2048,"indexName":"mflix_plots","isByo":true,"metric":"cosine","searchIndexName":"movies_vector_index","textSearchIndexName":null}


{"_id":"reviews","allowWrites":true,"collectionName":"movie_reviews","dimension":1024,"indexName":"reviews","isByo":true,"metric":"cosine","searchIndexName":"movie_reviews_vector_index","textSearchIndexName":null}


Note the entry from notebook 01 alongside this one. The registry is per-database and
outlives any single notebook. Its `_id` is the logical index name, which is why
`listIndexes()` returns logical names rather than collection names: a `listIndexes` →
`deleteIndex` round trip has to classify BYO correctly, and only the logical name can.

The registry lives in `__mastra_vector_indexes__`, deliberately *not* in your operational
collection. Your documents get an embedding field and nothing else.

## 6. What `deleteIndex` deletes

For a managed index, `deleteIndex` drops the collection: Mastra created it, so removing
the index means removing the data.

For a bring-your-own index it must not, and does not. It drops the Atlas search index and
removes the registry entry. Your collection and every document in it survive.

In [12]:
const beforeCount = await db.collection(REVIEWS).countDocuments();

await reviews.deleteIndex({ indexName: 'reviews' });

const collectionsNamed = await db.listCollections({ name: REVIEWS }).toArray();
console.log('Documents before deleteIndex:', beforeCount);
console.log('Documents after deleteIndex :', await db.collection(REVIEWS).countDocuments());
console.log('Collection still exists     :', collectionsNamed.length === 1);
const remainingIndexes = await db.collection(REVIEWS).listSearchIndexes().toArray();
console.log('Atlas search indexes left   :', remainingIndexes.map((i) => i.name));
const remainingRegistry = await db.collection('__mastra_vector_indexes__').find({}).toArray();
console.log('Registry entries            :', remainingRegistry.map((e) => String(e._id)));

Documents before deleteIndex: 12


Documents after deleteIndex : 12


Collection still exists     : true


Atlas search indexes left   : []


Registry entries            : [ 'mflix_plots' ]


Twelve documents in, twelve documents out. The collection is intact and the search index
is gone.

**But the search index really is gone**, and that is the sharp edge. On a BYO index the
Atlas index is usually a piece of your infrastructure, provisioned by a migration,
shared by other consumers, possibly by other Mastra logical indexes. `deleteIndex` drops
it, and rebuilding it on a large collection is not quick.

Notebook 01 declined to call `deleteIndex` for exactly this reason: its registration
pointed at `movies_vector_index`, which the other notebooks in this repository depend on.
Dropping it would have left every subsequent query returning nothing, with no visible
cause.

Mastra does check one case: an Atlas index still referenced by another logical index in
the registry is preserved. That covers two logical indexes sharing one Atlas index. It
cannot cover consumers outside Mastra, because it cannot see them.

The rule worth remembering: **on a BYO index, treat `deleteIndex` as a change to your
infrastructure, not to your Mastra configuration.** To stop using an index, drop the
registry entry and leave Atlas alone.

## Cleanup

We created a collection and an Atlas index, so we remove both, leaving the cluster as
notebook 01 left it, and freeing the third index slot for notebook 03.

In [13]:
await reviews.disconnect();
await db.collection(REVIEWS).drop();
await client.close();

console.log('Dropped the movie_reviews collection.');
console.log('sample_mflix and its two search indexes are untouched.');

Dropped the movie_reviews collection.


sample_mflix and its two search indexes are untouched.


## Recap

You pointed Mastra at a collection it did not create, with field names it did not choose
and `ObjectId` keys, and queried it without copying anything.

The three things worth carrying forward:

- **`collectionName` is the switch.** Passing it means bring-your-own, which means Mastra
  will not create or drop that collection.
- **Writes are refused by default.** `allowWrites` exists, but it writes Mastra's managed
  document shape into your collection. Prefer letting your application own writes.
- **`deleteIndex` on a BYO index still drops the Atlas index.** The collection is safe;
  the index is not.

**Next:** `03_hybrid_search_and_reranking.ipynb`. Semantic search misses exact terms:
a title, a proper noun, an error code. Full-text search catches those and misses meaning.
`$rankFusion` runs both and fuses the rankings server-side.